# Module 05: Generalization, Validation, & Regularization


## Module outline

* **Step 1: Generalization and Validation Foundations**
  * Generalization: why training score is not enough.
  * Train/test split: learning on one part, checking on unseen data.
  * Data leakage and target leakage: when the model accidentally sees future information or the answer.
  * Overfitting vs. underfitting: reading training and validation errors.
* **Step 2: Performance Metrics**
  * Regression metrics: MAE, MSE, RMSE, R-squared, Adjusted R-squared, and MAPE.
  * Classification metrics: confusion matrix, accuracy, precision, recall, F1-score, Macro/Micro/Weighted F1, log-loss, ROC-AUC, and PR-AUC.
* **Step 3: Cross-Validation Basics**
  * K-Fold Cross-Validation.
  * Stratified K-Fold for classification.
  * Group K-Fold for grouped records.
  * Time-Series Validation for ordered data.
* **Step 4: Regularization as an Overfitting Control**
  * Why penalties control model complexity.
  * L2/Ridge, L1/Lasso, and ElasticNet at concept level.

Detailed hyperparameter tuning and probability calibration are taught after the supervised algorithms, because each algorithm has different settings and probability behavior.


## Example Dataset


In [1]:
import pandas as pd
from IPython.display import display

# Let's define our exact 10-employee dataset in Python to display it cleanly
data = {
    'Employee ID': ['Emp 1', 'Emp 2', 'Emp 3', 'Emp 4', 'Emp 5', 'Emp 6', 'Emp 7', 'Emp 8', 'Emp 9', 'Emp 10'],
    'Department': ['Sales', 'R&D', 'HR', 'R&D', 'Sales', 'R&D', 'HR', 'Sales', 'R&D', 'Sales'],
    'Performance Rating': ['Low', 'Medium', 'Medium', 'High', 'High', 'Low', 'High', 'Medium', 'High', 'Medium'],
    'Years at Company': [1, 3, 2, 5, 4, 1, 8, 3, 10, 3],
    'Monthly Salary': [3000, 4500, 4000, 6000, 5500, 3200, 7000, 4800, 12000, 5000],
    'Attrition (Left?)': ['Yes (1)', 'No (0)', 'No (0)', 'No (0)', 'No (0)', 'Yes (1)', 'No (0)', 'No (0)', 'No (0)', 'Yes (1)']
}

df = pd.DataFrame(data)
display(df)


,Employee ID,Department,Performance Rating,Years at Company,Monthly Salary,Attrition (Left?)
0,Emp 1,Sales,Low,1,3000,Yes (1)
1,Emp 2,R&D,Medium,3,4500,No (0)
2,Emp 3,HR,Medium,2,4000,No (0)
3,Emp 4,R&D,High,5,6000,No (0)
4,Emp 5,Sales,High,4,5500,No (0)
5,Emp 6,R&D,Low,1,3200,Yes (1)
6,Emp 7,HR,High,8,7000,No (0)
7,Emp 8,Sales,Medium,3,4800,No (0)
8,Emp 9,R&D,High,10,12000,No (0)
9,Emp 10,Sales,Medium,3,5000,Yes (1)


## Module 5: Generalization, Validation, and Regularization

This module answers one question:

```text
Will the model work well on new data?
```

A model is not useful just because it performs well on the same rows it learned from. It must work on future or unseen rows.

In our employee dataset, we may build models for two different tasks:

- **Regression task:** predict `Monthly Salary`, a number.
- **Classification task:** predict `Attrition (Left?)`, a class: leave or stay.

The validation ideas are the same, but the metrics are different for regression and classification.


### Module 5: Step 1 - Generalization and Validation Foundations

#### 1. Generalization

- **Definition:** Generalization means the model learned a pattern that works on new unseen data.
- **Where we use it:** Every ML model that will be used after training.
- **Why we use it:** A model can memorize training rows and still fail on new rows.

Example:

```text
Training rows: current 10 employees
Future rows: new employees next month
```

If the model only memorizes the 10 current employees, it has not learned a useful rule.

#### 2. Train/Test Split

A train/test split separates data into two parts:

- **Training data:** used to teach the model.
- **Test data:** kept aside to check the model after training.

Correct order:

```text
split data -> fit preprocessing on train -> train model -> evaluate on test
```

The test set should behave like future unseen data.

#### 3. Data Leakage

- **Definition:** Data leakage happens when information from outside the training process enters the model or evaluation.
- **Simple meaning:** the model gets help that it would not have in real life.

Example with preprocessing:

```text
Wrong: calculate salary scaling using all employees, then split train/test.
Correct: split first, learn scaling from train only, apply it to test.
```

Why? If test data helped create the scaler, then the test set is no longer fully unseen.

#### 4. Target Leakage

- **Definition:** Target leakage happens when an input feature secretly contains the answer or future information related to the answer.
- **Simple meaning:** the model is not learning a real prediction rule; it is seeing a clue from the future.

In our employee attrition dataset, suppose we add these extra columns:

| Column | Can we use it to predict future attrition? | Why |
|---|---|---|
| `Monthly Salary` | yes | known before prediction |
| `Years at Company` | yes | known before prediction |
| `Performance Rating` | yes, if recorded before prediction | available before prediction |
| `Exit Interview Done` | no | exists only after employee leaves |
| `Final Settlement Amount` | no | exists after resignation |
| `Last Working Date` | no | directly reveals that the employee left |

The question to ask is:

```text
At prediction time, would this column already be known?
```

If not, it should not be used.

#### 5. Overfitting and Underfitting

| Problem | Meaning | Training score | Validation score |
|---|---|---:|---:|
| Underfitting | model is too simple | low | low |
| Good fit | model learned useful pattern | good | good |
| Overfitting | model memorized training details | very high | lower |

**Employee example:** A very deep decision tree may memorize all 10 employees. It can score perfectly on training data, but fail on new employees because it learned row-specific details instead of a general rule.


### Module 5: Step 2 - Regression Metrics

Regression means predicting a number. In our dataset, predicting `Monthly Salary` is a regression task.

Suppose we predicted salaries for four employees:

| Employee | Actual Salary | Predicted Salary | Error |
|---|---:|---:|---:|
| A | 3000 | 3200 | +200 |
| B | 4500 | 4300 | -200 |
| C | 5000 | 4500 | -500 |
| D | 7000 | 7600 | +600 |

The error is:

```text
prediction - actual
```

#### 1. MAE - Mean Absolute Error

- **Definition:** Average of absolute errors.
- **Formula:** `mean(abs(actual - predicted))`
- **Meaning:** On average, how many salary units are we wrong by?
- **Why useful:** Easy to explain because it uses the original unit, here salary.

#### 2. MSE - Mean Squared Error

- **Definition:** Average of squared errors.
- **Formula:** `mean((actual - predicted)^2)`
- **Meaning:** Large mistakes become much more costly.
- **Why useful:** Good when large errors are especially bad.

#### 3. RMSE - Root Mean Squared Error

- **Definition:** Square root of MSE.
- **Formula:** `sqrt(MSE)`
- **Meaning:** Like MSE, but returned to the original unit.
- **Why useful:** Penalizes large errors but is easier to read than MSE.

#### 4. R-squared

- **Definition:** Measures how much variation in the target is explained by the model.
- **Range:** Often between 0 and 1, but can be negative for a bad model.
- **Meaning:** Higher is usually better, but it does not show the size of salary error directly.

#### 5. Adjusted R-squared

- **Definition:** R-squared adjusted for the number of features.
- **Why useful:** Adding more features can increase normal R-squared even if they are not truly useful. Adjusted R-squared penalizes unnecessary features.

#### 6. MAPE - Mean Absolute Percentage Error

- **Definition:** Average percentage error.
- **Formula:** `mean(abs(actual - predicted) / actual) * 100`
- **Meaning:** Average error as a percentage.
- **Careful:** MAPE becomes unstable when actual values are zero or very close to zero.


In [2]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

salary_metric_example = pd.DataFrame({
    'Employee': ['A', 'B', 'C', 'D'],
    'Actual Salary': [3000, 4500, 5000, 7000],
    'Predicted Salary': [3200, 4300, 4500, 7600]
})
salary_metric_example['Error'] = salary_metric_example['Predicted Salary'] - salary_metric_example['Actual Salary']
salary_metric_example['Absolute Error'] = salary_metric_example['Error'].abs()
salary_metric_example['Squared Error'] = salary_metric_example['Error'] ** 2
salary_metric_example['Absolute Percentage Error'] = salary_metric_example['Absolute Error'] / salary_metric_example['Actual Salary'] * 100

y_true = salary_metric_example['Actual Salary']
y_pred = salary_metric_example['Predicted Salary']
mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred)
mape = salary_metric_example['Absolute Percentage Error'].mean()

# Adjusted R-squared example with p = 1 predictor and n = 4 rows.
n = len(y_true)
predictor_count = 1
adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - predictor_count - 1)

print('Salary prediction errors:')
display(salary_metric_example)

print('\nRegression metric summary:')
display(pd.DataFrame({
    'Metric': ['MAE', 'MSE', 'RMSE', 'R-squared', 'Adjusted R-squared', 'MAPE'],
    'Value': [mae, mse, rmse, r2, adjusted_r2, mape]
}))


Salary prediction errors:


,Employee,Actual Salary,Predicted Salary,Error,Absolute Error,Squared Error,Absolute Percentage Error
0,A,3000,3200,200,200,40000,6.666667
1,B,4500,4300,-200,200,40000,4.444444
2,C,5000,4500,-500,500,250000,10.000000
3,D,7000,7600,600,600,360000,8.571429



Regression metric summary:


,Metric,Value
0,MAE,375.000000
1,MSE,172500.000000
2,RMSE,415.331193
3,R-squared,0.915725
4,Adjusted R-squared,0.873588
5,MAPE,7.420635


### Module 5: Step 2 - Classification Metrics

Classification means predicting a category. In our dataset, predicting `Attrition (Left?)` is classification.

Let the positive class be:

```text
Positive = employee leaves
Negative = employee stays
```

#### Confusion Matrix

A confusion matrix compares actual classes with predicted classes.

| Term | Meaning in attrition |
|---|---|
| TP | predicted leave, actually left |
| TN | predicted stay, actually stayed |
| FP | predicted leave, actually stayed |
| FN | predicted stay, actually left |

#### Accuracy

- **Definition:** Fraction of all predictions that are correct.
- **Formula:** `(TP + TN) / all predictions`
- **Where useful:** Balanced classes.
- **Problem:** Can be misleading when one class is much larger.

#### Precision

- **Definition:** Out of employees predicted to leave, how many actually left?
- **Formula:** `TP / (TP + FP)`
- **Where useful:** When false alarms are costly.

#### Recall

- **Definition:** Out of employees who actually left, how many did we catch?
- **Formula:** `TP / (TP + FN)`
- **Where useful:** When missing a positive case is costly.

#### F1-score

- **Definition:** Balance between precision and recall.
- **Formula:** `2 * precision * recall / (precision + recall)`
- **Where useful:** Imbalanced classification when both false alarms and misses matter.

F1 averaging:

- **Macro F1:** average class scores equally.
- **Micro F1:** count all predictions together.
- **Weighted F1:** average class scores using class size.

#### Log-loss

- **Definition:** Measures probability quality. Wrong confident predictions get heavily punished.
- **Where useful:** When the model outputs probabilities, not just classes.

#### ROC-AUC and PR-AUC

- **ROC-AUC:** measures how well the model ranks positives above negatives across thresholds.
- **PR-AUC:** focuses on precision and recall; useful when positive cases are rare.

For attrition, PR-AUC can be more informative when very few employees leave.


In [3]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, log_loss, roc_auc_score, average_precision_score
)
import pandas as pd
import numpy as np

classification_example = pd.DataFrame({
    'Employee': ['Emp A', 'Emp B', 'Emp C', 'Emp D', 'Emp E', 'Emp F'],
    'Actual Left?': [1, 0, 1, 0, 0, 1],
    'Predicted Left?': [1, 0, 0, 0, 1, 1],
    'Predicted Probability of Leaving': [0.85, 0.20, 0.40, 0.10, 0.70, 0.65]
})

y_true = classification_example['Actual Left?']
y_pred = classification_example['Predicted Left?']
y_prob = classification_example['Predicted Probability of Leaving']

print('Attrition classification example:')
display(classification_example)

print('\nConfusion matrix:')
display(pd.DataFrame(
    confusion_matrix(y_true, y_pred, labels=[1, 0]),
    index=['Actual Leave', 'Actual Stay'],
    columns=['Predicted Leave', 'Predicted Stay']
))

print('\nClassification metric summary:')
display(pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1', 'Log-loss', 'ROC-AUC', 'PR-AUC'],
    'Value': [
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred, zero_division=0),
        recall_score(y_true, y_pred, zero_division=0),
        f1_score(y_true, y_pred, zero_division=0),
        log_loss(y_true, y_prob, labels=[0, 1]),
        roc_auc_score(y_true, y_prob),
        average_precision_score(y_true, y_prob)
    ]
}))


Attrition classification example:


,Employee,Actual Left?,Predicted Left?,Predicted Probability of Leaving
0,Emp A,1,1,0.85
1,Emp B,0,0,0.20
2,Emp C,1,0,0.40
3,Emp D,0,0,0.10
4,Emp E,0,1,0.70
5,Emp F,1,1,0.65



Confusion matrix:


,Predicted Leave,Predicted Stay
Actual Leave,2,1
Actual Stay,1,2



Classification metric summary:


,Metric,Value
0,Accuracy,0.666667
1,Precision,0.666667
2,Recall,0.666667
3,F1,0.666667
4,Log-loss,0.507012
5,ROC-AUC,0.777778
6,PR-AUC,0.805556


### Module 5: Step 3 - Cross-Validation Basics

- **Definition:** Cross-validation means checking a model across multiple train/validation splits.
- **Where we use it:** Model evaluation when one split may be too lucky or unlucky.
- **Why we use it:** It gives a more stable estimate of performance.

#### 1. K-Fold Cross-Validation

Split data into `k` parts.

```text
Fold 1: validate on part 1, train on the rest
Fold 2: validate on part 2, train on the rest
Fold 3: validate on part 3, train on the rest
```

The final score is the average across folds.

#### 2. Stratified K-Fold

Stratified K-Fold keeps class proportions similar in each fold.

Use it for classification, especially if `Left` is much smaller than `Stayed`.

#### 3. Group K-Fold

Group K-Fold keeps the same group from appearing in both train and validation.

Example:

```text
If one employee has many rows, all rows for that employee should stay in one side of the split.
```

#### 4. Time-Series Validation

Time-series validation respects time order.

Correct idea:

```text
train on past -> validate on future
```

Do not train on future data and validate on past data.

#### What is intentionally not deep here?

Detailed hyperparameter tuning is not taught deeply in this module. It becomes meaningful after learning algorithms, because each algorithm has different hyperparameters.


In [4]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, GroupKFold, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd

X = df[['Years at Company', 'Monthly Salary']]
y = df['Attrition (Left?)'].apply(lambda x: 1 if 'Yes' in x else 0)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='f1')

print('Stratified K-Fold F1 scores:')
display(pd.DataFrame({'Fold': [1, 2, 3], 'F1 Score': cv_scores}))
print(f'Mean F1: {cv_scores.mean():.3f}')

# Group K-Fold mechanics example. Groups are artificial for teaching.
groups = ['A', 'A', 'B', 'B', 'C', 'C', 'D', 'D', 'E', 'E']
gkf = GroupKFold(n_splits=5)
print('\nGroup K-Fold split groups:')
for fold, (train_idx, valid_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    train_groups = sorted(set(pd.Series(groups).iloc[train_idx]))
    valid_groups = sorted(set(pd.Series(groups).iloc[valid_idx]))
    print(f'Fold {fold}: train groups={train_groups}, validation groups={valid_groups}')

# Time-series split mechanics example.
ts = TimeSeriesSplit(n_splits=3)
print('\nTimeSeriesSplit index ranges:')
for fold, (train_idx, valid_idx) in enumerate(ts.split(X), start=1):
    print(f'Fold {fold}: train={train_idx.tolist()}, validation={valid_idx.tolist()}')


Stratified K-Fold F1 scores:


,Fold,F1 Score
0,1,0.666667
1,2,1.000000
2,3,0.000000


Mean F1: 0.556

Group K-Fold split groups:
Fold 1: train groups=['A', 'B', 'C', 'D'], validation groups=['E']
Fold 2: train groups=['A', 'B', 'C', 'E'], validation groups=['D']
Fold 3: train groups=['A', 'B', 'D', 'E'], validation groups=['C']
Fold 4: train groups=['A', 'C', 'D', 'E'], validation groups=['B']
Fold 5: train groups=['B', 'C', 'D', 'E'], validation groups=['A']

TimeSeriesSplit index ranges:
Fold 1: train=[0, 1, 2, 3], validation=[4, 5]
Fold 2: train=[0, 1, 2, 3, 4, 5], validation=[6, 7]
Fold 3: train=[0, 1, 2, 3, 4, 5, 6, 7], validation=[8, 9]


### Module 5: Step 4 - Regularization as Overfitting Control

- **Definition:** Regularization is a way to control model complexity by penalizing large weights.
- **Where we use it:** Linear regression, logistic regression, and many other models.
- **Why we use it:** To reduce overfitting.

Simple idea:

```text
Fit the data, but do not let the model weights become unnecessarily large.
```

#### L2 / Ridge

- **Meaning:** Penalizes squared weights.
- **Effect:** Shrinks weights smoothly toward smaller values.
- **Use when:** many features may be useful and we want stability.

Formula idea:

```text
normal loss + alpha * sum(weight^2)
```

#### L1 / Lasso

- **Meaning:** Penalizes absolute weights.
- **Effect:** Can push some weights exactly to zero.
- **Use when:** we want automatic feature selection.

Formula idea:

```text
normal loss + alpha * sum(abs(weight))
```

#### ElasticNet

- **Meaning:** Combines L1 and L2.
- **Effect:** Can remove weak features while still staying stable with correlated features.

This module teaches the purpose of regularization. Algorithm-specific regularization settings are easier to understand after the algorithms are introduced.


In [5]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Regression-style target only for demonstrating coefficient shrinkage.
X_reg = df[['Years at Company']]
y_reg = df['Monthly Salary']

models = {
    'Ridge_L2': Ridge(alpha=10.0),
    'Lasso_L1': Lasso(alpha=100.0, max_iter=10000),
    'ElasticNet_L1_L2': ElasticNet(alpha=100.0, l1_ratio=0.5, max_iter=10000)
}

rows = []
for name, model in models.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    pipe.fit(X_reg, y_reg)
    fitted = pipe.named_steps['model']
    rows.append({
        'Model': name,
        'Coefficient_for_Years': fitted.coef_[0],
        'Intercept': fitted.intercept_
    })

print('Regularization coefficient comparison:')
display(pd.DataFrame(rows))


Regularization coefficient comparison:


,Model,Coefficient_for_Years,Intercept
0,Ridge_L2,1165.476957,5500.0
1,Lasso_L1,2230.953915,5500.0
2,ElasticNet_L1_L2,44.724587,5500.0
